In [ ]:
!pip install "google-tunix[prod]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of numba to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.0/488.0 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.4/194.4 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.9/310.9 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.3/504.3 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

## Importing Modules

In [ ]:
import os
import re
import json
import logging
from flax import nnx
from huggingface_hub import snapshot_download
import jax
import jax.numpy as jnp

from datasets import load_dataset,concatenate_datasets
import optax
from tunix.generate import tokenizer_adapter as tokenizer_lib

from tunix.models.gemma3 import params_safetensors as params_safetensors_lib
from tunix.models.gemma3 import model as gemma_lib
# from tunix.rl.experimental.agentic_grpo_learner import GRPOConfig, GRPOLearner
from flax import nnx
from tunix.cli.utils import model as model_utils

from orbax import checkpoint as ocp
import qwix

from tunix.sft import metrics_logger
from tunix.sft import peft_trainer
from tunix.sft import utils

logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Gemma 3 1B
model_id = "google/gemma-3-1b"


/usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.8.1, so it will not be used.
  warnings.warn(


## Configurations

In [ ]:
# Configs

# Data
BATCH_SIZE = 64
MAX_TARGET_LENGTH = 1024 # Adjusted based on your TPU memory and model size.

# Model Setup
# Adjust mesh based on your TPU memory and model size.
NUM_TPUS = len(jax.devices())
if NUM_TPUS == 8:
  MESH_COUNTS = (1, 4)
elif NUM_TPUS == 1:
  MESH_COUNTS = (1, 1)
else:
  raise ValueError(f"Unsupported number of TPUs: {NUM_TPUS}")

MESH = [
    MESH_COUNTS,
    ("fsdp", "tp"),
]

# Need for GRPO
# TEMPERATURE = 0.9
# TOP_P = 1.0  # implies we don't do nucleus sampling
# TOP_K = 50

# The number of iterations per batch (𝜇 in GRPO algo 1).
# NUM_ITERATIONS = 1
# EPSILON = 0.2


# LoRA/QLoRA Configuration
USE_QUANTIZATION = True
RANK = 16
ALPHA = 2.0

# Train
MAX_STEPS = 2500
EVAL_EVERY_N_STEPS = 20
NUM_EPOCHS = 3

# Checkpoint saving
# FULL_CKPT_DIR = "/tmp/content/full_ckpts/"
LORA_CKPT_DIR = "/tmp/content/lora_ckpts/"


In [ ]:
# Save metadata for loading
checkpoint_metadata = {
    "base_model": model_id,
    "lora_rank": RANK,
    "lora_alpha": ALPHA,
    "quantization": USE_QUANTIZATION,
    "output_format": "<reasoning>...</reasoning><answer>...</answer>"
}

## BASE MODEL LOADING...

In [ ]:

ignore_patterns = [
    "*.pth",  # Ignore PyTorch .pth weight files
]
print(f"Downloading {model_id} from Hugging Face...")
local_model_path = snapshot_download(
    repo_id=model_id, ignore_patterns=ignore_patterns
)
print(f"Model successfully downloaded to: {local_model_path}")

In [ ]:
MODEL_CP_PATH = local_model_path

if "gemma-3-1b" in model_id:
  model_config = gemma_lib.ModelConfig.gemma3_1b()
else:
  raise ValueError(f"Unsupported model: {model_id}")

mesh = jax.make_mesh(*MESH, axis_types=(jax.sharding.AxisType.Auto,) * len(MESH[0]))

with mesh:
  base_model = params_safetensors_lib.create_model_from_safe_tensors(
      MODEL_CP_PATH, (model_config), mesh
  )
  # nnx.display(base_model) just calls cluster output

## MODEL TOKENIZATION

In [ ]:
# initialize tokenizer
GEMMA_TOKENIZER_PATH = "gemma-data/tokenizers/tokenizer_gemma3.model"
tokenizer = tokenizer_lib.Tokenizer(tokenizer_path=GEMMA_TOKENIZER_PATH)
EOS_TOKENS = [tokenizer.eos_id()]

print(f"Using EOS token IDs: {EOS_TOKENS}")

In [ ]:

# def create_dir(path):
#   try:
#     os.makedirs(path, exist_ok=True)
#     logging.info(f"Created dir: {path}")
#   except OSError as e:
#     logging.error(f"Error creating directory '{path}': {e}")


# # create_dir(FULL_CKPT_DIR)
# create_dir(LORA_CKPT_DIR)


## SYSTEM PROMPT

In [ ]:
SYSTEM_PROMPT = """You are a helpful AI assistant that thinks step by step.
When answering questions:
1. First, provide your reasoning inside <reasoning></reasoning> tags
2. Then, provide your final answer inside <answer></answer> tags"""


## LoRA TRAINER

In [ ]:
def use_lora(base_model,mesh,quantize:bool=False):
  if quantize:
    lora_provider = qwix.LoraProvider(
        module_path=".*q_einsum|.*kv_einsum|.*gate_proj|.*down_proj|.*up_proj",
        rank=RANK,
        alpha=ALPHA,
        weight_qtype="nf4",
        tile_size=128,
    )
  else:
    lora_provider = qwix.LoraProvider(
        module_path=".*q_einsum|.*kv_einsum|.*gate_proj|.*down_proj|.*up_proj",
        rank=RANK,
        alpha=ALPHA,
    )
  model_input = base_model.get_model_input()
  lora_model = qwix.apply_lora_to_model(
      base_model, lora_provider, **model_input
  )

  with mesh:
    state = nnx.state(lora_model)
    pspecs = nnx.get_partition_spec(state)
    sharded_state = jax.lax.with_sharding_constraint(state, pspecs)
    nnx.update(lora_model, sharded_state)

  return lora_model


In [ ]:
# Create LoRA or QLoRA model based on USE_QUANTIZATION hyperparameter
lora_model = use_lora(base_model, mesh=mesh, quantize=USE_QUANTIZATION)

print(f"Using {'QLoRA' if USE_QUANTIZATION else 'LoRA'} model")

## RESPONSE EXTRACTION

In [ ]:
def extract_final_number(answer_text):
    """
    Extract the final numerical answer from GSM8K answer format
    GSM8K answers end with "#### NUMBER"
    """
    import re

    # GSM8K format: "reasoning steps\n#### 42"
    match = re.search(r'####\s*([0-9,\.]+)', answer_text)
    if match:
        return match.group(1).replace(',', '')  # Remove commas

    # Fallback: try to find last number
    numbers = re.findall(r'-?\d+\.?\d*', answer_text)
    if numbers:
        return numbers[-1]

    return "0"  # Fallback if no number found

In [ ]:
def extract_answer(answer_text, domain:str="math"):
    """Extract answer based on domain"""
    if domain == "math":
        return extract_final_number(answer_text)
    elif domain == "coding":
        # Extract code block
        return answer_text  # Adjust as needed
    else:
        # For creative/summarization, use full text
        return answer_text

## CHAT TEMPLATE

In [ ]:
def chat_template(raw_text):
    """Add reasoning format to responses"""
    prompt = raw_text['prompt']
    response = raw_text['response']
    domain = raw_text.get('domain', 'general')

    # Create reasoning wrapper
    # For simplicity, use entire response as reasoning
    # and extract key part as answer

    # Extract answer based on domain
    if domain == 'math':
        # Try to get numerical answer
        answer = extract_final_number(response)
    elif domain == 'code':
      code_blocks = re.findall(r'```.*?```', response, re.DOTALL)
      if code_blocks:
          answer = code_blocks[-1]  # Last code block
      else:
          answer = response[:200]  # First 200 chars
    else:
        # For text tasks, use last sentence or summary
        sentences = response.split('.')
        answer = sentences[-1].strip() if sentences else response[:100]

    formatted_response = f"""<reasoning>
{response}
</reasoning>
<answer>
{answer}
</answer>"""

    messages = [
        {
            "role": "user",
            "content": prompt
        },
        {
            "role": "assistant",
            "content": formatted_response
        }
    ]

    return {"messages": messages}

## GENERATION OF ATTENTION AND PADDING TOKENS

In [ ]:
def gen_model_input_fn(batch):
  """
  Args: Takes formatted text and tokenizes with padding and creates attention masks
  Returns : model inputs
  """
  texts = batch['text']
  tokens = []

  for text in texts:
    # Tokenize with EOS
    token_ids = tokenizer.encode(text)
    if not text.endswith(tokenizer.eos_token):
        token_ids.append(tokenizer.eos_id())

    # Truncate if needed
    if len(token_ids) > MAX_TARGET_LENGTH:
        token_ids = token_ids[:MAX_TARGET_LENGTH]

    tokens.append(token_ids)

  # Pad to same length
  max_len = max(len(t) for t in tokens)
  padded_tokens = []
  attention_masks = []


  for token_ids in tokens:
      padding_length = max_len - len(token_ids)

        # Pad with pad token
      padded = token_ids + [tokenizer.pad_id()] * padding_length
      mask = [1] * len(token_ids) + [0] * padding_length

      padded_tokens.append(padded)
      attention_masks.append(mask)


  return {
        'input_ids': jnp.array(padded_tokens, dtype=jnp.int32),
        'attention_mask': jnp.array(attention_masks, dtype=jnp.int32),
    }

## REBALANCING DATASET

In [ ]:
def prepare_datasets_with_chat_template():
    """
    Load and REBALANCE multi-domain dataset
    """
    # Load full dataset
    ds = load_dataset("Addyk24/Multi-domain-reasoning")

    # Define sampling strategy
    SAMPLE_SIZES = {
        "math": None,
        "coding": None,
        "science": None,
        "general": 12000,
        "summarization": 10000,
        "creative": 4000,
    }

    # Rebalance training set
    print("Rebalancing dataset...")
    train_balanced = []
    val_balanced = []

    for domain, max_size in SAMPLE_SIZES.items():
        # Filter by domain
        domain_data = ds['train'].filter(lambda x: x['domain'] == domain)

        if max_size and len(domain_data) > max_size:
            # Sample down
            domain_data = domain_data.shuffle(seed=42).select(range(max_size))
            print(f"  {domain}: {len(domain_data)} (sampled from more)")
        else:
            # Keep all
            print(f"{domain}: {len(domain_data)} (kept all)")

        train_balanced.append(domain_data)

    # Also balance validation set proportionally
    for domain in SAMPLE_SIZES.keys():
        domain_val = ds['test'].filter(lambda x: x['domain'] == domain)
        if len(domain_val) > 0:
            # Take up to 10% for validation or max 1000 per domain
            val_size = min(len(domain_val), 1000)
            domain_val = domain_val.shuffle(seed=42).select(range(val_size))
            val_balanced.append(domain_val)

    # Concatenate and shuffle
    from datasets import concatenate_datasets
    train_ds = concatenate_datasets(train_balanced).shuffle(seed=42)
    val_ds = concatenate_datasets(val_balanced).shuffle(seed=42)

    print(f"\Balanced dataset:")
    print(f"   Training: {len(train_ds)} examples")
    print(f"   Validation: {len(val_ds)} examples")

    # Format to message structure
    train_ds = train_ds.map(chat_template, remove_columns=train_ds.column_names)
    val_ds = val_ds.map(chat_template, remove_columns=val_ds.column_names)

    # Apply chat template
    def apply_template(example):
        formatted_text = tokenizer.apply_chat_template(
            example['messages'],
            add_generation_prompt=False,
            tokenize=False
        )
        return {"text": formatted_text}

    train_ds = train_ds.map(apply_template)
    val_ds = val_ds.map(apply_template)

    print(f"\n Final counts:")
    print(f"   Training: {len(train_ds)}")
    print(f"   Validation: {len(val_ds)}")
    print("\n   Example:")
    print(train_ds[0]['text'][:300] + "...")

    return train_ds, val_ds

<>:51: SyntaxWarning: invalid escape sequence '\B'
<>:51: SyntaxWarning: invalid escape sequence '\B'
/tmp/ipython-input-4210513313.py:51: SyntaxWarning: invalid escape sequence '\B'
  print(f"\Balanced dataset:")


In [ ]:
# train_ds, validation_ds = prepare_datasets_with_chat_template()


In [ ]:
# def prepare_datasets_with_chat_template():
#     """
#     Load GSM8K and format using apply_chat_template
#     """
#     # Load dataset
#     ds = load_dataset("Addyk24/Multi-domain-reasoning")

#     # Format to message structure
#     train_ds = ds['train'].map(
#         chat_template,
#         remove_columns=ds['train'].column_names
#     )

#     validation_ds = ds['test'].map(
#         chat_template,
#         remove_columns=ds['test'].column_names
#     )

#     # Apply chat template to each example
#     def apply_template(example):
#         # Get formatted text using tokenizer's chat template
#         formatted_text = tokenizer.apply_chat_template(
#             example['messages'],
#             add_generation_prompt=False,  # We already have assistant response
#             tokenize=False  # Return string, not tokens
#         )
#         return {"text": formatted_text}

#     train_ds = train_ds.map(apply_template)
#     validation_ds = validation_ds.map(apply_template)

#     print(f"Training examples: {len(train_ds)}")
#     print(f"Validation examples: {len(validation_ds)}")

#     # Show example
#     print("\n\tExample Formatted Training Data\t")
#     print(train_ds[0]['text'][:500] + "...")

#     return train_ds, validation_ds

## POST TRAINING BASE MODEL - Gemma-3n-1B

In [ ]:

# Getting train, validation dataset
train_ds, validation_ds = prepare_datasets_with_chat_template()

lora_logging_options = metrics_logger.MetricsLoggerOptions(
    log_dir="/tmp/tensorboard/lora", flush_every_n_steps=20
)

training_config = peft_trainer.TrainingConfig(
    eval_every_n_steps=EVAL_EVERY_N_STEPS,
    max_steps=MAX_STEPS,
    metrics_logging_options=lora_logging_options,
    checkpoint_root_directory=LORA_CKPT_DIR,
)

trainer = peft_trainer.PeftTrainer(
    lora_model, optax.adamw(1e-3), training_config
).with_gen_model_input_fn(gen_model_input_fn)

# The first couple of training step might take up to 5 minutes to finish. Please be patient. If you experience long training steps, e.g. >10 minutes per step, please open a bug. Really appreciated!
method_name = "QLoRA" if USE_QUANTIZATION else "LoRA"
with mesh:
    trainer.train(train_ds, validation_ds)


In [ ]:
# After training completes
print("\n" + "="*60)
print("CHECKPOINT SAVED")
print("="*60)
print(f"Checkpoint location: {LORA_CKPT_DIR}")
print(f"Base model: {model_id}")
print(f"LoRA config: rank={RANK}, alpha={ALPHA}")
print(f"Quantization: {USE_QUANTIZATION}")
print("\nTo load this checkpoint:")
print("1. Load base model from Hugging Face")
print("2. Apply LoRA with same config")
print("3. Load checkpoint weights from:", LORA_CKPT_DIR)
print("="*60)

# Save metadata
with open(os.path.join(LORA_CKPT_DIR, "training_metadata.json"), "w") as f:
    json.dump(checkpoint_metadata, f, indent=2)

## Inferenece

In [ ]:
def generate_reasoning_response(prompt:str, max_tokens:int=512):
  """
      Generate response with reasoning format
    Args:
        prompt: User query string
        max_tokens: Maximum tokens to generate
    Returns:
        Generated response string
  """

  messages = [{"role": "user", "content": prompt}]

  # Format prompt
  formatted_prompt = tokenizer.apply_chat_template(
      messages,
      add_generation_prompt=True,
      tokenize=False,
  )

  # Tokenize
  input_ids = tokenizer.encode(formatted_prompt)
  input_len = len(input_ids)
  # Convert to JAX array with proper shape
  input_ids_array = jnp.array([input_ids], dtype=jnp.int32)

  # Generation of outputs
  with mesh:

    output_ids = lora_model.generate(
      input_ids=input_ids_array,
      max_new_tokens=max_tokens,
      temperature=0.7,
      top_p=0.9,
      eos_token_ids=EOS_TOKENS,
    )

    # Decode only the generated portion (skip input)
    generated_ids = output_ids[0][input_len:]  # Remove input tokens
    response = tokenizer.decode(generated_ids.tolist())

    return response


# Test after training
print("\n" + "="*60)
print("TESTING MODEL")
print("="*60)

test_prompts = [
    "What is 25% of 80?",
    "Explain photosynthesis in simple terms",
    "Write a haiku about coding"
]

for test_prompt in test_prompts:
    print(f"\nPrompt: {test_prompt}")
    try:
        response = generate_reasoning_response(test_prompt, max_tokens=512)
        print(f"Response:\n{response}\n")
        print("-" * 60)
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
# Training dataset composition:
# Maths : 	8,790 (Use All)	100%
# Creative : 9510	16%
# Reasoning : 	15,000	28.84%
# Science : 	11,700 (Use All)	100%
# Summarization : 10,000	16.6%
# Coding : 	5,000 (Use All)	100%
# Total : 	50,000	100%

# Validation dataset composition:
# General Reasoning 1,500	30%
# Maths 750	15%
# Science	500	10%
# Coding  500 (All)	100%
# Summarization	750	15%
# Creative	1,000	20%
# TOTAL		5,000	100%